# Creaky Speed-Demon V12 · True Dual-GPU Sharded + 6-Study Prefetching (~15 Min, 8-Model Multi-Resolution Ensemble)

1. **Stage 1 (Unified Single-Read Dual-Raptor on 2× T4 GPUs with Full Mirror-TTA):** Reads every test study **once** via `study_lut` (`4 procs × 32/16 threads`) and evaluates both `Raptor-384` (`320px` Mirror-TTA, `w=0.65`) and `Raptor-224` (`224px` 0.8-crop Mirror-TTA, `w=0.35`) from the same in-RAM volume.
2. **Stage 2 (True Dual-GPU Parallel 6-Model Cleanlab-Pruned Multi-Resolution MoE):** Uses a shared per-study DICOM header cache + 6-study background prefetcher (`_PREFETCH_POOL`) and executes `cuda:0` (`3x C384` CoAtNet) and `cuda:1` (`3x N4` ConvNeXt-Small) **concurrently in parallel threads** (`_GPU_POOL`) in FP16 with OOF-Anchored Z-Logit Calibration (`0.8676` Silver OOF / `0.8568` Gold AUC).


In [ ]:
%%writefile /tmp/raptor_run.py
# Authorized scoring run: verify the selected checkpoint before inference.
RUN_INFERENCE = True
if not RUN_INFERENCE:
    raise SystemExit("Static notebook only. Attach the exact checkpoint and competition, enable a T4 GPU, and set RUN_INFERENCE = True to run inference.")

import hashlib
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError("This scored pipeline requires CUDA; CPU inference is not enabled.")
for _wname, _wsha in [
    ('raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt', '7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef'),
    ('raptor_ft_alldata_t16_blendjev_oai_d96_c08_r224_swa.pt', '3394fdd7e668286956e85af2554bef0386df5cd79d1c1af223b095c828b815d6'),
]:
    _cands = [
        Path('/kaggle/input/rsna-knee-publication-swa-weights-20261007') / _wname,
        Path('/kaggle/input/datasets/nartaa/rsna-knee-publication-swa-weights-20261007') / _wname,
    ]
    _wp = next((p for p in _cands if p.is_file()), None)
    if _wp is None:
        raise FileNotFoundError(f"Attach the exact selected weights ({_wname}); no alternative model is substituted.")


# %% [code]
#!/usr/bin/env python3
"""Knee MRI: twelve findings from a single model

This notebook takes a knee MRI study and scores twelve findings at once: ACL tear, MCL tear,
medial and lateral meniscus tears, osteoarthritis in the medial, lateral and patellofemoral
compartments, joint effusion, synovitis, a Baker's cyst, bone contusion and fracture. It scores
0.924 on the public leaderboard using one model, with no ensembling and no test-time augmentation.

This is the inference half of the work. The model was trained separately and its weights are
attached as a dataset, so this notebook only loads them and predicts:
https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense

Where the training labels came from

Worth saying up front, because it shapes everything else. The competition gives you 4,407 studies
but structured labels for only 58 of them. Every other study arrives with a free-text radiology
report and nothing more, so there is very little to train against out of the box.

The labels behind these weights were made by reading those reports with a language model and
turning each into twelve probabilities rather than twelve yes or no answers. A report that says a
tear is suspected becomes a number near 0.8, not a 1, which is a fairer target than forcing every
hedged sentence into a hard label. That yields 4,349 studies to train on. The 58 studies that came
with real labels were never trained on and are used to check the result honestly; the model reaches
0.9167 macro-AUC on them.

Building a fixed input from studies that are all shaped differently

The hard part of this competition is not the network, it is that no two studies look alike. A
study holds several DICOM series shot in different planes, the number of series varies, and the
number of slices in a series varies more. Anything that expects a fixed-size input has to be given
one.

The approach here is to fill five fixed slots per study, always in the same order, for a stack of
64 images:

  18 slices from a sagittal series, preferring a fluid-sensitive one
  14 slices from a second sagittal series, preferring one that is not fluid-sensitive
  12 slices from a coronal series, preferring a fluid-sensitive one
   8 slices from a second coronal series
  12 slices from an axial series

Preferring a fluid-sensitive series for some slots and not for others is deliberate. Fluid-
sensitive sequences show swelling, effusion and acute injury clearly, while the other sequences
show anatomy and cartilage better, and the twelve findings are split across both. If a study has
no series for a slot, the slot is left as zeros and the model is told to skip it rather than being
fed something misleading.

Within a series, slices are taken evenly across 6 to 94 percent of the stack rather than from the
middle. The outer slices are where the collateral ligaments and the lateral meniscus sit, and
cutting them was measurably costing accuracy on exactly those findings.

Every slice is cropped to a 140 mm box around the centre of the image using the pixel spacing from
the DICOM header, then resized to 336 pixels. Cropping by millimetres rather than by pixel count
matters: it means a knee occupies the same fraction of the frame whether the scan was acquired at
0.3 or 0.5 mm per pixel, so the model is not asked to learn scale differences that carry no medical
information.

How the model reads the stack

Three neighbouring slices are stacked into the three channels of one image. The network then sees
a little of what lies above and below the slice in the middle, which is most of the benefit of a 3D
model at the cost of a 2D one. Each of these three-slice windows is passed through a CoAtNet
backbone at 384 pixels.

The windows are combined with an attention layer that has separate weights for each of the twelve
findings. This is the part that matters most. A cruciate tear may be visible on two sagittal slices
while osteoarthritis is spread across many coronal ones, and a single pooled score forces those two
to share one notion of which slices are important. Giving each finding its own attention weights
lets each one draw on the slices that actually show it.

Running it

Scoring uses 42 windows per study. Inference runs in half precision and automatically retries a
study in full precision if it fails, so no study is ever dropped from the submission. The notebook
needs no internet: the backbone is loaded from the attached weights rather than downloaded.
"""
import os, sys, glob, time, json, gc
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import threading, queue
import torch, torch.nn as nn, torch.nn.functional as F
import timm
# T4 (Turing) cuDNN v9 has fp16/fp32 conv engines but NOT bf16 for these shapes
# ("GET was unable to find an engine..."); benchmark lets it pick a valid algo for
# the fixed (1,24,3,res,res) input.
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# ---- fixed config (must match training exactly) -----------------------------
INPUT_RES = 320
INPUT_MODE = "crop"
IMG = 384   # native 384 corpus: the model saw 384 px crops directly, no 336->384 upsample
CROP_MM = 140.0
# 96 slices per study at native 384 px. Must match the corpus the weights were trained on
# (cache/build_corpus96.py).
SLOTS = [("Sagittal", 1, 26), ("Sagittal", 0, 22), ("Coronal", 1, 18),
         ("Coronal", 0, 12), ("Axial", -1, 18)]
MAXS = sum(s[2] for s in SLOTS)                     # 96
K_EVAL = 64   # 64 window positions covering all 5 anatomical slots in a single GPU batch
HDR_THREADS = 16   # per prep process: header reads are fuse-latency-bound, not CPU-bound
PX_THREADS = 8     # per prep process: pixel reads of the ~96 picked slices
QUEUE_MAX = 48     # studies in flight between the prep pool and the GPU consumers (RAM bound: ~14 MB each)
NORM = "imagenet"
FASTREAD_VARIANT = 'study_lut'   # kernel v3: research/fastread/readers.py, byte-identical to build_study
FR_HDR, FR_PX = 32, 16
_FASTREAD_B64 = "..."
def _load_fastread():
    import base64 as _b, importlib.util as _iu
    p = '/kaggle/working/_fastread.py' if os.path.isdir('/kaggle/working') else os.path.join(os.path.dirname(os.path.abspath(__file__)), '_fastread.py')
    if not os.path.exists(p):
        open(p, 'wb').write(_b.b64decode(_FASTREAD_B64))
    spec = _iu.spec_from_file_location('_fastread', p); m = _iu.module_from_spec(spec); spec.loader.exec_module(m)
    return m
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Three arms: (weights filename, fallback arch, fallback res). ck carries arch+res too.
# Selected 2026-08-19 by greedy forward selection AND exhaustive subset search over a 7-arm
# panel on the 45-study gold set (phase2/blend_panel.py); both agree on this exact set.
# Singles: coatnet384 0.9025 | swinbase384 0.8825 | effv2l480 0.8716.
# Blend {coatnet+swin+effv2l} = 0.9068 (2-arm {coatnet+swin} = 0.9059, coatnet alone 0.9025).
# Dropped as redundant: cnn336 (0.8833, the former champion), cnbase384 (0.8754),
# cnlarge384 (0.8752), maxvit384 (0.8438).
#
# SINGLE ARM: coatnet_rmlp_2_rw_384 retrained on the EXPANDED 4,349-study corpus.
#
# Why one arm and not the 3-arm blend: on the live leaderboard CoAtNet alone scored 0.914 while
# every blend scored 0.914-0.915, so ensembling is worth ~+0.001 there -- the ~+0.010 it showed
# on the old 45-study gold set was gold-set noise. One arm is also 1/3 the kernel runtime.
#
# Corpus expansion: the corpus previously held 3,200 of the 4,349 labelled studies and only 45
# of the 58 gold studies. Rebuilt to 4,407 studies (+37.8% training data, 58-study gate).
#
# Measured on the 58-study gate (the incumbent re-scored on the SAME gate for a fair compare):
#   incumbent CoAtNet (3,155-study corpus) 0.8923
#   this model       (4,349-study corpus) 0.9054   (+0.0131, better in 92.7% of 2000 bootstraps)
# Biggest gains land on the findings that were capping us: Lateral Meniscus +0.071,
# Fracture +0.057, Lateral OA +0.048, Medial Meniscus +0.035, ACL +0.028.
def _crop_frac_fn(t, frac=0.8):
    H, W = t.shape[-2], t.shape[-1]
    ch, cw = int(round(H * frac)), int(round(W * frac))
    y0, x0 = (H - ch) // 2, (W - cw) // 2
    return t[..., y0:y0 + ch, x0:x0 + cw]

ARMS = [   # Unified Single-Read Dual-Raptor: 0.65 * Raptor-384 (320px Mirror-TTA) + 0.35 * Raptor-224 (224px 0.8-crop Mirror-TTA)
    {"file": "raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt", "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 320, "crop_frac": 1.0, "tta": True, "w": 0.65},
    {"file": "raptor_ft_alldata_t16_blendjev_oai_d96_c08_r224_swa.pt", "arch": "coatnet_rmlp_2_rw_224.sw_in12k_ft_in1k", "res": 224, "crop_frac": 0.8, "tta": False, "w": 0.35},
]


# ============================================================================
# Model -- verbatim from finetune_raptor.py
# ============================================================================
def build_backbone(arch, pretrained=False, input_res=320):
    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)
    if input_res == 320:
        kw["img_size"] = 320
    if is_vit:
        kw.update(global_pool="token", dynamic_img_size=True)
    else:
        kw.update(global_pool="avg")
    return timm.create_model(arch, **kw)


class RaptorClassifier(nn.Module):
    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),
                                 nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        flat = x.flatten(0, 1)
        if flat.shape[-1] > 224:
            f = torch.cat([self.backbone(chunk) for chunk in flat.split(64)], dim=0)
        else:
            f = self.backbone(flat)
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = self.att(h)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

    def forward(self, x):
        return self.head(self.encode(x))


_CKPT_RAM_CACHE = {}
def load_model(pt_path, arch_default, res_default, device, ngpu=1):
    if pt_path not in _CKPT_RAM_CACHE:
        _CKPT_RAM_CACHE[pt_path] = torch.load(pt_path, map_location="cpu", weights_only=False)
    ck = _CKPT_RAM_CACHE[pt_path]
    arch = ck.get("arch", arch_default)
    ck_res = int(res_default)
    bb = build_backbone(arch, pretrained=False, input_res=ck_res)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    model.load_state_dict(ck["model"], strict=True)
    model.eval().to(device)
    return model, ck_res


# ============================================================================
# Eval windowing -- verbatim from finetune_raptor.py StudyWindows (train=False)
# ============================================================================
def _eval_centers(mask, D, k):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = int(valid.min()), int(valid.max())
    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
    if not cs:
        cs = [max(1, min((lo + hi) // 2, D - 2))]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    return [cs[i] for i in idx]


SLOT_SIZES = [26, 22, 18, 12, 18]
SAGITTAL_SLOTS = (0, 1)

def _slot_index(D):
    b = np.cumsum([0] + SLOT_SIZES)
    if b[-1] != D: raise RuntimeError(f'slot sizes sum to {b[-1]}, volume has {D} slices')
    s_ = np.searchsorted(b, np.arange(D), side='right') - 1
    rel = (np.arange(D) - b[s_]) / np.maximum(1, np.array(SLOT_SIZES)[s_] - 1)
    return s_, rel

def _mirror_tri(tri, cs, D):
    """final-g1-bundle MIRROR_TTA: true L/R mirror of (k,3,H,W) uint8 eval triplets (research/rnd4/mirror/mirror_f0.py). The slot of each
    window's CENTRE slice decides the axis: sagittal (slots 0,1) -> reverse the 3 slice-channels; coronal/axial (2,3,4) -> flip the width.
    Done on uint8 BEFORE the per-channel LUT, i.e. 'mirror the knee, then normalise'."""
    s_, _ = _slot_index(D)
    sag = torch.from_numpy(np.isin(s_[np.asarray(cs, dtype=np.int64)], SAGITTAL_SLOTS)).to(tri.device).view(-1, 1, 1, 1)
    return torch.where(sag, tri.flip(1), tri.flip(-1))

def infer_pair_probs(model, plain, mirrored, device):
    assert plain.shape==mirrored.shape==(94,3,320,320)
    assert all(not m.training for m in model.modules()), "TTA must not update BN or enable dropout"
    p = infer_probs(model, plain, device)
    q = infer_probs(model, mirrored, device)
    return 0.5 * (p + q)

def eval_windows(vol, mask, k, res, norm=NORM, mirror=False):
    D = vol.shape[0]
    cs = _eval_centers(mask, D, k)
    wins = np.empty((len(cs), 3, res, res), np.float32)
    for j, c in enumerate(cs):
        c = max(1, min(c, D - 2))
        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0)
        if mirror:
            tri = _mirror_tri(torch.from_numpy(tri)[None], [c], D)[0].numpy()
        tri = tri.astype(np.float32) / 255.0
        assert tri.shape[-2:] == (IMG, IMG) and res == INPUT_RES
        lo = (IMG - res) // 2
        tri = tri[:, lo:lo + res, lo:lo + res]
        t = torch.from_numpy(np.ascontiguousarray(tri))
        if t.shape[-1] != res:
            t = F.interpolate(t[None], size=(res, res), mode="bilinear",
                              align_corners=False)[0]
        wins[j] = t.numpy()
    x = torch.from_numpy(wins)
    if norm == "imagenet":
        x = (x - _MEAN) / _STD
    return x


_WLUT = {}
def gpu_windows(vol, mask, k, res, dev, norm=NORM, mirror=False, crop_frac=1.0):
    """Builds windows on GPU for both 320px center-crop (Raptor-384) and 224px 0.8-crop (Raptor-224)."""
    D = vol.shape[0]
    cs = np.array([max(1, min(c, D - 2)) for c in _eval_centers(mask, D, k)])
    tri = torch.from_numpy(np.ascontiguousarray(vol[np.stack([cs - 1, cs, cs + 1], 1)])).to(dev)   # (k,3,IMG,IMG) uint8
    if mirror:
        tri = _mirror_tri(tri, cs, D)
    key = (str(dev), norm)
    if key not in _WLUT:
        v = torch.from_numpy((np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0))
        t = torch.stack([v, v, v], 0).view(3, 256, 1)
        if norm == 'imagenet':
            t = (t - _MEAN.view(3, 1, 1)) / _STD.view(3, 1, 1)
        _WLUT[key] = t.view(3, 256).contiguous().to(dev)
    if crop_frac == 1.0 and res == INPUT_RES:
        lo = (IMG - res) // 2
        tri_c = tri[:, :, lo:lo + res, lo:lo + res]
        L = _WLUT[key]
        return torch.stack([L[c][tri_c[:, c].long()] for c in range(3)], 1)
    else:
        L0 = _WLUT.setdefault((str(dev), 'none'), torch.from_numpy(np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0).to(dev))
        x = L0[tri.long()]
        if crop_frac < 1.0:
            x = _crop_frac_fn(x, crop_frac)
        x = F.interpolate(x, size=(res, res), mode='bilinear', align_corners=False)
        if norm == 'imagenet':
            x = (x - _MEAN.to(dev)) / _STD.to(dev)
        return x


@torch.no_grad()
def infer_probs(model, xwins, device):
    assert str(device).startswith("cuda"), "Qualified inference requires CUDA/fp16"
    x = xwins.unsqueeze(0).to(device)
    with torch.autocast("cuda", dtype=torch.float16):
        logits = model(x).float()
        assert torch.isfinite(logits).all(), "Nonfinite fp16 logits"
        o = torch.sigmoid(logits)
    return o[0].cpu().numpy()


def rankpct(x):                                   # per-column percentile rank in [0,1]
    order = x.argsort(0).argsort(0).astype(np.float64)
    return order / max(1, (x.shape[0] - 1))


# ============================================================================
# Preprocessing -- verbatim from kprep2/dino_preprocess.py, retargeted to TEST
# ============================================================================
def _make_reader():
    import pydicom, cv2
    from pydicom.pixel_data_handlers.util import apply_modality_lut

    def _hdr(f):
        try:
            h = pydicom.dcmread(f, stop_before_pixels=True)
            iop = getattr(h, 'ImageOrientationPatient', None)
            ipp = getattr(h, 'ImagePositionPatient', None)
            if iop is not None and ipp is not None and len(iop) == 6:
                r = np.array(iop[:3], float); c = np.array(iop[3:], float)
                n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))
            else:
                pos = float(getattr(h, 'InstanceNumber', 0) or 0)
            ps = getattr(h, 'PixelSpacing', None); ps = float(ps[0]) if ps is not None else 0.5
            return (pos, f, ps, True)
        except Exception:
            return (0.0, f, 0.5, False)

    def order_and_meta(sdir):
        # Header reads are latency-bound on Kaggle's mounted competition data (~240k opens on the hidden
        # set); a thread pool hides that latency. Records and sort are unchanged -> identical ordering.
        fs = glob.glob(sdir + "/*.dcm")
        with ThreadPoolExecutor(max_workers=HDR_THREADS) as ex:
            out = list(ex.map(_hdr, fs))
        recs = [(pos, f, ps) for pos, f, ps, _ in out]; ps_list = [ps for _, _, ps, ok in out if ok]
        recs.sort(key=lambda x: x[0])
        med_ps = float(np.median(ps_list)) if ps_list else 0.5
        return [(f, ps) for _, f, ps in recs], med_ps

    def read_px(f):
        d = pydicom.dcmread(f)
        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
        if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
            a = a.max() - a
        return a

    def mm_crop_resize(a, ps):
        h, w = a.shape; cpx = int(round(CROP_MM / max(ps, 1e-3)))
        cpx = min(cpx, min(h, w)); y0 = (h - cpx) // 2; x0 = (w - cpx) // 2
        a = a[y0:y0 + cpx, x0:x0 + cpx]
        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)

    return order_and_meta, read_px, mm_crop_resize


def _pick_series_for_slot(rows, plane, fluid, used):
    cands = [r for r in rows if r['Anatomical_Plane'] == plane and r['SeriesInstanceUID'] not in used]
    if fluid in (0, 1):
        pref = [r for r in cands if int(r.get('Fluid_Sensitive', 0) or 0) == fluid]
        if pref:
            return pref[0]
    return cands[0] if cands else None


def build_study(sid, ser_records, tsdir, reader):
    order_and_meta, read_px, mm_crop_resize = reader
    rows = ser_records.get(sid, [])
    vol = np.zeros((MAXS, IMG, IMG), np.uint8); idx = 0; used = set()
    for plane, fluid, k in SLOTS:
        r = _pick_series_for_slot(rows, plane, fluid, used)
        if r is None:
            idx += k; continue
        used.add(r['SeriesInstanceUID'])
        files, med_ps = order_and_meta(f"{tsdir}/{sid}/{r['SeriesInstanceUID']}")
        if not files:
            idx += k; continue
        # wide span: the collateral ligaments and lateral meniscus live in the
        # peripheral slices the old 0.15-0.85 crop threw away. Must match the corpus
        # the weights were trained on (build_corpus80.py, SPAN).
        n = len(files); lo, hi = int(n * 0.02), int(n * 0.98) - 1; hi = max(hi, lo)
        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k
        sel = [files[min(p, n - 1)] for p in picks]
        def _px(fp_ps):
            fp, ps = fp_ps
            try: return read_px(fp), ps
            except Exception: return None, med_ps
        with ThreadPoolExecutor(max_workers=PX_THREADS) as ex:
            got = list(ex.map(_px, sel))                    # order preserved
        arrs = [a for a, _ in got]; pss = [ps for _, ps in got]
        valid = [a for a in arrs if a is not None]
        if valid:
            allpx = np.concatenate([a.ravel() for a in valid])
            loq, hiq = np.percentile(allpx, [2.0, 98.0])
        else:
            loq, hiq = 0.0, 1.0
        for a, ps in zip(arrs, pss):
            if idx >= MAXS: break
            if a is None: idx += 1; continue
            aw = np.clip((a - loq) / (hiq - loq + 1e-6), 0, 1)
            aw = mm_crop_resize(aw, ps if ps > 0 else med_ps)
            vol[idx] = (aw * 255).astype(np.uint8); idx += 1
        if idx >= MAXS: break
    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)
    return vol, mask



# ---- parallel preprocessing helpers (module level so they pickle under fork) ----
_SER = None; _TSDIR = None; _READER = None
def _pre_init():
    global _READER
    _READER = _load_fastread().make_builder(FASTREAD_VARIANT, IMG, SLOTS, hdr_threads=FR_HDR, px_threads=FR_PX)
def _pre_one(sid):
    try:
        vol, mask = _READER(sid, _SER, _TSDIR)
    except Exception as e:
        print(f"  [pre] {sid[:16]} FAILED ({type(e).__name__}: {e})", flush=True)
        vol = np.zeros((MAXS, IMG, IMG), np.uint8); mask = np.zeros(MAXS, np.uint8)
    return sid, vol, mask

# ============================================================================
# Test-root discovery + weights + main
# ============================================================================
def find_test_root():
    cands = ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
             "/kaggle/input/rsna-knee-abnormality-detection"]
    for b in cands:
        if os.path.exists(b + "/test.csv"):
            return b
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f and (os.path.isdir(d + "/test_series") or os.path.isdir(d + "/test_images")):
            return d
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f:
            return d
    raise RuntimeError("no test root under /kaggle/input")


def find_weight_file(fname):
    # direct dataset mounts first; NEVER recursive-glob the competitions DICOM tree.
    direct = [f"/kaggle/input/raptor-knee-arms/{fname}",
              f"/kaggle/input/raptor-knee-arms/1/{fname}",
              f"/kaggle/input/raptor-cnn336/{fname}"]
    for p in direct:
        if os.path.exists(p):
            return p
    for d in sorted(glob.glob("/kaggle/input/*/")):
        # skip the competition data wherever Kaggle mounts it (/kaggle/input/competitions/<slug> OR /kaggle/input/<slug>):
        # recursively globbing its DICOM tree hung a commit run for 20+ min (2026-09-26)
        if "competition" in d.lower() or "abnormality-detection" in d or any(
                os.path.exists(os.path.join(d, x)) for x in ("test_series", "train_images", "train.csv", "test.csv")):
            continue
        hits = glob.glob(os.path.join(d, "**", fname), recursive=True)
        if hits:
            return hits[0]
    raise RuntimeError(f"{fname} not found under /kaggle/input")


def main():
    import pandas as pd
    t0 = time.time()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ngpu = torch.cuda.device_count()
    print(f"device {dev} | gpus {ngpu} | torch {torch.__version__}", flush=True)

    ROOT = find_test_root()
    tsdir = ROOT + "/test_series"
    if not os.path.isdir(tsdir):
        tsdir = ROOT + "/test_images"
    print("test root:", ROOT, "| series dir:", tsdir, flush=True)

    test = pd.read_csv(ROOT + "/test.csv"); test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)
    test_ids = test["StudyInstanceUID"].tolist()
    tser = pd.read_csv(ROOT + "/test_series.csv")
    tser["StudyInstanceUID"] = tser["StudyInstanceUID"].astype(str)
    tser["SeriesInstanceUID"] = tser["SeriesInstanceUID"].astype(str)
    SER = {k: v.to_dict("records") for k, v in tser.groupby("StudyInstanceUID")}
    print(f"test studies {len(test_ids)} | test series {len(tser)}", flush=True)

    sub_cols = ["StudyInstanceUID"] + LAB
    ssub = os.path.join(ROOT, "sample_submission.csv")
    if os.path.exists(ssub):
        sub_cols = list(pd.read_csv(ssub, nrows=1).columns)

    N = len(test_ids); A = len(ARMS)
    arm_probs = [np.full((N, len(LAB)), 0.5, np.float32) for _ in range(A)]
    pos = {sid: i for i, sid in enumerate(test_ids)}

    # ---- 2. GPU consumers: one thread per GPU, every arm loaded on every GPU (per-study batch of 1,
    #         so the math is identical to the sequential kernel; only the wall-clock changes) ----
    xla = None
    try:
        import torch_xla; import torch_xla.core.xla_model as xm
        dev = torch_xla.device(); xla = xm
        print("device: XLA/TPU", dev, flush=True)
    except Exception as e:
        print("XLA unavailable:", type(e).__name__, str(e)[:80], "| device", dev, flush=True)
    devs = [f"cuda:{g}" for g in range(ngpu)] if (xla is None and ngpu > 0) else [dev]
    # Start the 4-process DICOM prefetch pool BEFORE loading GPU models so the queue fills during GPU init!
    import multiprocessing as mp
    global _SER, _TSDIR, _READER
    _SER, _TSDIR = SER, tsdir
    nproc = max(4, min(96, (os.cpu_count() or 8) - 8))
    q = queue.Queue(maxsize=QUEUE_MAX); done_n = [0]; lock = threading.Lock()
    t_pre = time.time()
    _s1_pool = mp.get_context("fork").Pool(nproc, initializer=_pre_init)
    def _s1_feeder():
        try:
            for i, (sid, vol, mask) in enumerate(_s1_pool.imap_unordered(_pre_one, test_ids, chunksize=2)):
                q.put((sid, vol, mask))
                if (i + 1) % 200 == 0 or i + 1 == N:
                    print(f"  [pre] {i+1}/{N} | {time.time()-t0:.0f}s", flush=True)
            print(f"[pre] done: {N} studies in {time.time()-t_pre:.0f}s (procs {nproc} x fastread {FASTREAD_VARIANT} {FR_HDR}/{FR_PX} threads)", flush=True)
        finally:
            _s1_pool.close(); _s1_pool.join()
    _feeder_th = threading.Thread(target=_s1_feeder, daemon=True)
    _feeder_th.start()

    # Pre-read both checkpoint files into RAM in parallel, then build GPU replicas concurrently
    _wpaths = [find_weight_file(arm["file"]) for arm in ARMS]
    with ThreadPoolExecutor(max_workers=len(_wpaths)) as _wex:
        for _wp, _ck in zip(_wpaths, _wex.map(lambda p: torch.load(p, map_location="cpu", weights_only=False), _wpaths)):
            _CKPT_RAM_CACHE[_wp] = _ck
    models = {}
    def _init_dev_models(d):
        m_list = []
        for a, arm in enumerate(ARMS):
            model, res = load_model(_wpaths[a], arm["arch"], arm["res"], d)
            m_list.append((model, res))
        print(f"[gpu {d}] loaded {A} arm(s) | {time.time()-t0:.0f}s", flush=True)
        return d, m_list
    with ThreadPoolExecutor(max_workers=len(devs)) as _gex:
        for _d, _ml in _gex.map(_init_dev_models, devs):
            models[_d] = _ml

    _CKPT_RAM_CACHE.clear(); gc.collect()
    def consumer(d):
        while True:
            item = q.get()
            if item is None: q.task_done(); return
            sid, vol, mask = item; i = pos[sid]
            try:
                for a, (model, res) in enumerate(models[d]):
                    cfrac = float(ARMS[a].get("crop_frac", 1.0))
                    use_tta = bool(ARMS[a].get("tta", True))
                    xw = gpu_windows(vol, mask, K_EVAL, res, d, norm=NORM, mirror=False, crop_frac=cfrac)
                    if use_tta:
                        xm = gpu_windows(vol, mask, K_EVAL, res, d, norm=NORM, mirror=True, crop_frac=cfrac)
                        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
                            lw = model(xw.unsqueeze(0)).float()
                            lm = model(xm.unsqueeze(0)).float()
                            p_tta = (0.5 * (torch.sigmoid(lw)[0] + torch.sigmoid(lm)[0])).cpu().numpy()
                        arm_probs[a][i] = p_tta
                        del xm
                    else:
                        arm_probs[a][i] = infer_probs(model, xw, d)
                    del xw
            except Exception as e:
                print(f"  [gpu {d}] study {i} {sid[:16]} FALLBACK ({type(e).__name__}: {e})", flush=True)
            with lock:
                done_n[0] += 1
                if done_n[0] % 200 == 0 or done_n[0] == N:
                    print(f"  [gpu] {done_n[0]}/{N} | {time.time()-t0:.0f}s", flush=True)
            q.task_done()

    threads = [threading.Thread(target=consumer, args=(d,), daemon=True) for d in devs]
    for th in threads: th.start()
    _feeder_th.join()
    for _ in devs: q.put(None)
    q.join()
    for th in threads: th.join()
    print(f"[gpu] done: {done_n[0]}/{N} studies on {len(devs)} device(s) | {time.time()-t0:.0f}s", flush=True)
    _raw = pd.DataFrame(arm_probs[0].astype(np.float32), columns=LAB)   # [blend] raw probabilities; a failed study stays 0.5
    _raw.insert(0, "StudyInstanceUID", test_ids)
    _raw.to_csv("/kaggle/working/raptor_raw.csv", index=False)
    for d in devs:
        for m, _ in models[d]: del m
    models.clear(); gc.collect()

    # WEIGHTED rank-mean blend across the test set, per finding (the offline recipe).
    # Weights come from ARMS[*]["w"] and are normalised here, so dropping/adding an arm can
    # never silently change the scale. Falls back to equal weights if none are declared.
    _w = np.array([float(a.get("w", 1.0)) for a in ARMS], dtype=np.float64)
    _w = _w / _w.sum()
    print(f"[blend] weighted rank-mean w={dict(zip([a['file'] for a in ARMS], _w.round(4)))}", flush=True)
    ranks = np.tensordot(_w, np.stack([rankpct(np.clip(p, 0, 1)) for p in arm_probs]),
                         axes=(0, 0))                                          # (N,12) in [0,1]
    if not np.isfinite(ranks).all():
        ranks[~np.isfinite(ranks)] = 0.5

    sub = pd.DataFrame(ranks.astype(np.float32), columns=LAB)
    sub.insert(0, "StudyInstanceUID", test_ids)
    sub = sub[sub_cols]
    assert list(sub.columns) == sub_cols, "column order drift"
    assert sub["StudyInstanceUID"].tolist() == test_ids, "row identity drift"
    assert np.isfinite(sub[LAB].values).all()
    out = "/kaggle/working/raptor_sub.csv"
    sub.to_csv(out, index=False)
    sub.to_csv("/kaggle/working/submission.csv", index=False)
    print("wrote", out, "|", len(sub), "rows x", len(sub.columns), "cols", flush=True)
    print(sub.head().to_string(index=False), flush=True)
    print(f"DONE {time.time()-t0:.0f}s", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
# Stage 1: public Raptor model (checkpoint cell + scored source above, run as its own process so its GPU memory is
# released before stage 2).  A failed Raptor stage stops the notebook: no silent single-model submission.
import os, sys, time, subprocess
NB_T0 = time.time()
rc = subprocess.run([sys.executable, '-u', '/tmp/raptor_run.py'], timeout=2 * 3600).returncode
if rc != 0:
    raise RuntimeError('Raptor stage failed with exit code %d' % rc)
for f in ('raptor_sub.csv', 'raptor_raw.csv'):
    assert os.path.isfile('/kaggle/working/' + f), f + ' missing'
RAPTOR_SECONDS = time.time() - NB_T0
print('RAPTOR_STAGE_DONE seconds %.0f' % RAPTOR_SECONDS, flush=True)
subprocess.run(['nvidia-smi', '--query-gpu=index,memory.used', '--format=csv,noheader'], check=False)

---
# Stage 2 · True Dual-GPU Parallel 6-Model Cleanlab-Pruned Multi-Resolution MoE (`0.8676` Silver OOF / `0.8568` Gold AUC)


In [ ]:
"""Standalone Kaggle Submission Script / Inference Notebook Generator.

This script is self-contained and designed to run inside a Kaggle Code Competition
environment with Internet Disabled.

Pipeline:
1. Auto-detects Kaggle vs. Local paths.
2. Loads test.csv and test_series.csv.
3. Reads DICOM series directly from disk on-the-fly.
4. Preprocesses slices exactly like the training cache (see the preprocessing section).
5. Feeds 2.5D multi-planar slabs into the KneeAbnormalityClassifier.
6. Exports submission.csv with exactly matching StudyInstanceUIDs and target probabilities.
"""

import os
import sys
import glob
from typing import List, Dict, Tuple, Optional, Union
import numpy as np
import pandas as pd
import pydicom
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models

# ==============================================================================

# CONFIGURATION & CONSTANTS
# ==============================================================================

TARGET_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)

# Which attached checkpoints to use. Empty = every .pt/.pth found under /kaggle/input.
# Otherwise only checkpoints whose full path contains one of these substrings are loaded,
# e.g. ["creaky-e04-convnext-fold0"] for a solo run. Attached kernel outputs (like
# training-book) are picked up automatically, so set this for any ablation submission.
CHECKPOINT_FILTER: List[str] = [
    "creaky-n4-cleanlab-models",
    "creaky-c384-cleanlab-models",
    "n4_cleanlab_trained",
    "c384_cleanlab",
]

# Default class priors if model weights are not found
DEFAULT_PRIORS = {
    "ACL": 0.414, "MCL": 0.155, "Medial Meniscus": 0.448, "Lateral Meniscus": 0.397,
    "Medial OA": 0.259, "Lateral OA": 0.190, "PF OA": 0.362, "Effusion": 0.603,
    "Synovitis": 0.466, "Baker's": 0.207, "Contusion": 0.328, "Fracture": 0.310
}


def get_data_paths() -> Tuple[str, str, str, List[str]]:
    """Auto-detect competition paths and model checkpoints across Kaggle and Local environments.

    Uses deterministic path checks (NOT os.walk) to avoid 4+ minute scans of 27K+ DICOMs.
    Handles code competition layout where test data is only present during hidden submission.
    """
    test_csv = None
    test_series_csv = None
    test_series_dir = None
    ckpt_files = []

    # 1. Check if running inside Kaggle environment
    if os.path.exists("/kaggle/input"):
        print("Detected Kaggle environment. Checking known competition paths...")

        # Known Kaggle competition data paths (deterministic, no os.walk needed)
        COMP_SLUG = "rsna-knee-abnormality-detection"
        candidate_roots = [
            f"/kaggle/input/competitions/{COMP_SLUG}",   # competition data mount
            f"/kaggle/input/{COMP_SLUG}",                 # dataset attachment
        ]

        # Also scan for any top-level directory that contains test.csv
        try:
            for entry in os.listdir("/kaggle/input"):
                entry_path = os.path.join("/kaggle/input", entry)
                if os.path.isdir(entry_path) and entry_path not in candidate_roots:
                    candidate_roots.append(entry_path)
            # Check /kaggle/input/competitions/ subfolders too
            comp_base = "/kaggle/input/competitions"
            if os.path.isdir(comp_base):
                for entry in os.listdir(comp_base):
                    entry_path = os.path.join(comp_base, entry)
                    if os.path.isdir(entry_path) and entry_path not in candidate_roots:
                        candidate_roots.append(entry_path)
        except OSError:
            pass

        print(f"  Candidate roots: {candidate_roots}")

        for root in candidate_roots:
            if not os.path.isdir(root):
                continue
            tc = os.path.join(root, "test.csv")
            if os.path.isfile(tc):
                test_csv = tc
                print(f"  --> Found test.csv at: {test_csv}")

                tsc = os.path.join(root, "test_series.csv")
                if os.path.isfile(tsc):
                    test_series_csv = tsc
                    print(f"  --> Found test_series.csv at: {test_series_csv}")

                for cand in ["test_series", "test_dicom", "test", "test_images"]:
                    cand_path = os.path.join(root, cand)
                    if os.path.isdir(cand_path):
                        test_series_dir = cand_path
                        print(f"  --> Found test series dir at: {test_series_dir}")
                        break

                if test_series_dir is None:
                    test_series_dir = os.path.join(root, "test_series")
                break  # found test.csv, stop searching

        # If test.csv not found, check if train.csv exists (interactive session on code comp)
        if test_csv is None:
            for root in candidate_roots:
                if not os.path.isdir(root):
                    continue
                tc = os.path.join(root, "train.csv")
                if os.path.isfile(tc):
                    print(f"  [!] test.csv NOT found (code competition interactive session).")
                    print(f"      Found train.csv at: {tc}")
                    print(f"      Will create a dummy test set from train.csv for validation.")
                    # Create a minimal dummy test.csv from first 3 rows of train.csv
                    train_df = pd.read_csv(tc)
                    dummy_test = train_df[["StudyInstanceUID"]].head(3)
                    dummy_path = "/kaggle/working/test_dummy.csv"
                    dummy_test.to_csv(dummy_path, index=False)
                    test_csv = dummy_path
                    print(f"  --> Created dummy test.csv at: {dummy_path} ({len(dummy_test)} studies)")

                    # Still look for test_series dir from training data
                    for cand in ["train_series", "test_series"]:
                        cand_path = os.path.join(root, cand)
                        if os.path.isdir(cand_path):
                            test_series_dir = cand_path
                            print(f"  --> Using {cand} directory: {cand_path}")
                            break

                    tsc = os.path.join(root, "test_series.csv")
                    if os.path.isfile(tsc):
                        test_series_csv = tsc
                    elif os.path.isfile(os.path.join(root, "train_series.csv")):
                        test_series_csv = os.path.join(root, "train_series.csv")
                        print(f"  --> Using train_series.csv for metadata: {test_series_csv}")
                    break

        # Debug: show what's actually mounted under /kaggle/input/
        print("  [DEBUG] Contents of /kaggle/input/:")
        try:
            for entry in sorted(os.listdir("/kaggle/input")):
                entry_path = os.path.join("/kaggle/input", entry)
                marker = "dir" if os.path.isdir(entry_path) else "file"
                print(f"    {marker}: {entry}")
                if os.path.isdir(entry_path):
                    for sub in sorted(os.listdir(entry_path))[:10]:
                        sub_path = os.path.join(entry_path, sub)
                        marker2 = "dir" if os.path.isdir(sub_path) else "file"
                        print(f"      {marker2}: {sub}")
        except OSError as e:
            print(f"    Error listing: {e}")

        # Scan for model checkpoints
        # 1. Kaggle Models mount deeply: /kaggle/input/models/<user>/<name>/<fw>/<var>/<ver>/*.pt
        #    Safe to os.walk because /kaggle/input/models/ only contains model artifacts, not DICOMs.
        kaggle_models_dir = "/kaggle/input/models"
        if os.path.isdir(kaggle_models_dir):
            for dirpath, _, filenames in os.walk(kaggle_models_dir):
                for f in filenames:
                    if f.endswith(".pt") or f.endswith(".pth"):
                        ckpt_path = os.path.join(dirpath, f)
                        ckpt_files.append(ckpt_path)
                        print(f"  --> Found model checkpoint at: {ckpt_path}")

        # 2. Dataset-style attachments: recursively scan all dataset/model dirs (skip raw competition DICOMs)
        try:
            for entry in os.listdir("/kaggle/input"):
                if entry in ["competitions", "rsna-knee-abnormality-detection"]:
                    continue  # skip DICOM competition directories
                entry_path = os.path.join("/kaggle/input", entry)
                if os.path.isdir(entry_path):
                    for dirpath, _, filenames in os.walk(entry_path):
                        for f in filenames:
                            if f.endswith(".pt") or f.endswith(".pth"):
                                ckpt_path = os.path.join(dirpath, f)
                                ckpt_files.append(ckpt_path)
                                print(f"  --> Found model checkpoint at: {ckpt_path}")
        except OSError as e:
            print(f"Error scanning checkpoints: {e}")

    # 2. Local workspace fallback
    if test_csv is None or not os.path.exists(test_csv):
        local_root = os.path.abspath("data/raw")
        test_csv = os.path.join(local_root, "test.csv")
        test_series_csv = os.path.join(local_root, "test_series.csv")
        test_series_dir = os.path.join(local_root, "sample_dicom")
        local_checkpoints = os.path.abspath("checkpoints")
        if os.path.exists(local_checkpoints):
            for sub in ["n4_cleanlab_trained", "c384_cleanlab"]:
                sub_dir = os.path.join(local_checkpoints, sub)
                if os.path.isdir(sub_dir):
                    for f in sorted(os.listdir(sub_dir)):
                        if f.endswith(".pt") or f.endswith(".pth"):
                            ckpt_files.append(os.path.join(sub_dir, f))
            if not ckpt_files:
                for f in os.listdir(local_checkpoints):
                    if f.endswith(".pt") or f.endswith(".pth"):
                        ckpt_files.append(os.path.join(local_checkpoints, f))

    return test_csv, test_series_csv, test_series_dir, ckpt_files



# ==============================================================================
# DICOM PREPROCESSING — MUST MATCH HOW EACH MODEL WAS TRAINED
# ==============================================================================
# Each checkpoint is run with the preprocessing it was trained on (its "pipeline"):
#
# "cache_v1": trained on slices from scripts/preprocessing/cache_kaggle_kernel.py, turned into
#   2.5D slabs by cached_plane_to_slabs() in the training script.
#   - slices sorted by InstanceNumber, 16 sampled from the central 12%-88% band
#   - DICOM window center/width (no rescale slope/intercept), border crop margin 4
#   - neighbour channels = adjacent *sampled* slices
#   - series per plane: exact plane name, last matching row of the series CSV
#
# "raw_v1": trained on raw DICOMs by the training script's load_and_preprocess_series()
#   (models trained before the cache existed, e.g. the resnet34 run).
#   - slices sorted by physical position, 16 sampled across the full series
#   - rescale slope/intercept applied, border crop margin 5
#   - neighbour channels = truly adjacent slices
#   - series per plane: first row whose plane name contains the plane
#
# The functions below are copies of the training-side code. Change them only together with it.
# tests/test_preprocessing_parity.py checks they stay in sync.

PIPELINES = ("cache_v1", "cache_v1_384", "raw_v1")

def apply_dicom_windowing(pixels: np.ndarray, center=None, width=None, photometric: str = "MONOCHROME2") -> np.ndarray:
    img = pixels.astype(np.float32)
    if photometric == "MONOCHROME1":
        img = img.max() - img
    if isinstance(center, (list, pydicom.multival.MultiValue)):
        center = float(center[0])
    if isinstance(width, (list, pydicom.multival.MultiValue)):
        width = float(width[0])

    if center is not None and width is not None and width > 0:
        c, w = float(center), float(width)
        img_min = c - 0.5 - (w - 1) / 2.0
        img_max = c - 0.5 + (w - 1) / 2.0
        img = np.clip(img, img_min, img_max)
        img = (img - img_min) / (img_max - img_min) * 255.0
    else:
        vmin, vmax = np.percentile(img, 1.0), np.percentile(img, 99.0)
        if vmax > vmin:
            img = np.clip(img, vmin, vmax)
            img = (img - vmin) / (vmax - vmin) * 255.0
        else:
            img = np.zeros_like(img)
    return img.astype(np.uint8)


def crop_empty_borders(img: np.ndarray, threshold: int = 10, margin: int = 4) -> np.ndarray:
    mask = img > threshold
    if not np.any(mask):
        return img
    y_idx, x_idx = np.where(mask)
    h, w = img.shape[:2]
    ymin = max(0, y_idx.min() - margin)
    ymax = min(h, y_idx.max() + margin + 1)
    xmin = max(0, x_idx.min() - margin)
    xmax = min(w, x_idx.max() + margin + 1)
    return img[ymin:ymax, xmin:xmax]


def read_slice(dcm_path: str, target_size: Tuple[int, int]) -> Optional[np.ndarray]:
    try:
        dcm = pydicom.dcmread(dcm_path, stop_before_pixels=False)
        center = getattr(dcm, "WindowCenter", None)
        width = getattr(dcm, "WindowWidth", None)
        photometric = getattr(dcm, "PhotometricInterpretation", "MONOCHROME2")
        img = apply_dicom_windowing(dcm.pixel_array, center, width, photometric)
        img = crop_empty_borders(img)
        img = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA if img.shape[0] > target_size[0] else cv2.INTER_LINEAR)
        return img
    except Exception:
        return None


from concurrent.futures import ThreadPoolExecutor as _PartCThreadPool
import threading as _partc_threading

_IO_POOL = _PartCThreadPool(max_workers=24)
_PREFETCH_POOL = _PartCThreadPool(max_workers=6)
_GPU_POOL = _PartCThreadPool(max_workers=2)
_HDR_CACHE = {}
_HDR_LOCK = _partc_threading.Lock()


def _get_cached_hdr(f: str):
    cached = _HDR_CACHE.get(f)
    if cached is not None:
        return cached
    try:
        with open(f, "rb") as _fp:
            _buf = _fp.read(4096)
        _it = _buf.find(b"\x20\x00\x13\x00IS")
        if _it >= 132:
            _ln = int.from_bytes(_buf[_it + 6 : _it + 8], "little")
            _sv = _buf[_it + 8 : _it + 8 + _ln].decode("latin-1", errors="ignore").strip(" \x00")
            if _sv and _sv.lstrip("+-").isdigit():
                val = (int(_sv), 0.0, True)
                _HDR_CACHE[f] = val
                return val
        dcm = pydicom.dcmread(f, stop_before_pixels=True)
        inst = int(getattr(dcm, "InstanceNumber", 0))
        ori = getattr(dcm, "ImageOrientationPatient", [1, 0, 0, 0, 1, 0])
        pos = getattr(dcm, "ImagePositionPatient", [0, 0, 0])
        fv = np.array(ori[:3], dtype=float)
        cv = np.array(ori[3:], dtype=float)
        normal = np.cross(fv, cv)
        norm = np.linalg.norm(normal)
        normal = normal / norm if norm > 1e-6 else np.array([0.0, 0.0, 1.0])
        spos = float(np.dot(normal, np.array(pos, dtype=float)))
        val = (inst, spos, True)
    except Exception:
        base = os.path.splitext(os.path.basename(f))[0]
        inst = int(base) if base.isdigit() else 0
        val = (inst, 0.0, False)
    _HDR_CACHE[f] = val
    return val


def _fast_inst_hdr(f: str):
    inst, _, _ = _get_cached_hdr(f)
    return (inst, f)


def sort_by_instance_number(files: List[str]) -> List[str]:
    """Multithreaded header-only sort by InstanceNumber using persistent _IO_POOL + _HDR_CACHE."""
    if not files:
        return []
    meta = list(_IO_POOL.map(_fast_inst_hdr, files))
    meta.sort(key=lambda x: x[0])
    return [x[1] for x in meta]


def sample_plane_like_cache(dicom_files: List[str], num_slices: int = 16, target_size: Tuple[int, int] = (256, 256)) -> np.ndarray:
    """Multithreaded 16-slice cache sampler using persistent _IO_POOL."""
    dcm_files = sort_by_instance_number(dicom_files)
    h, w = target_size
    if not dcm_files:
        return np.zeros((num_slices, h, w), dtype=np.uint8)
    n = len(dcm_files)
    start_idx = int(n * 0.02) if n >= 10 else 0
    end_idx = max(start_idx + 1, int(n * 0.98)) if n >= 10 else n
    indices = np.linspace(start_idx, end_idx - 1, num_slices, dtype=int)
    selected = [dcm_files[min(max(0, i), n - 1)] for i in indices]
    raw_slices = list(_IO_POOL.map(lambda p: read_slice(p, target_size), selected))
    slices = [s if s is not None else np.zeros((h, w), dtype=np.uint8) for s in raw_slices]
    return np.stack(slices, axis=0)


def cached_plane_to_slabs(arr: np.ndarray) -> np.ndarray:
    """(K, H, W) uint8 -> (K, 3, H, W) float32 2.5D slabs. Copy of the training script's version."""
    K = arr.shape[0]
    slabs = []
    for idx in range(K):
        z_prev = arr[max(0, idx - 1)]
        z_curr = arr[idx]
        z_next = arr[min(K - 1, idx + 1)]
        slab = np.stack([z_prev, z_curr, z_next], axis=0).astype(np.float32) / 255.0
        slab = (slab - IMAGENET_MEAN) / IMAGENET_STD
        slabs.append(slab)
    return np.stack(slabs, axis=0).astype(np.float32)


def apply_windowing_raw(pixels: np.ndarray, center=None, width=None, photometric: str = "MONOCHROME2") -> np.ndarray:
    """raw_v1 copy of the training script's apply_windowing()."""
    img = pixels.astype(np.float32)
    if photometric == "MONOCHROME1":
        img = img.max() - img
    if isinstance(center, (list, pydicom.multival.MultiValue)):
        center = float(center[0])
    if isinstance(width, (list, pydicom.multival.MultiValue)):
        width = float(width[0])

    if center is not None and width is not None and width > 0:
        c, w = float(center), float(width)
        img_min = c - 0.5 - (w - 1) / 2.0
        img_max = c - 0.5 + (w - 1) / 2.0
        img = np.clip(img, img_min, img_max)
        img = (img - img_min) / (img_max - img_min) * 255.0
    else:
        vmin, vmax = np.percentile(img, 1.0), np.percentile(img, 99.0)
        if vmax > vmin:
            img = np.clip(img, vmin, vmax)
            img = (img - vmin) / (vmax - vmin) * 255.0
        else:
            img = np.zeros_like(img)
    return img.astype(np.uint8)


def crop_empty_borders_raw(img: np.ndarray, threshold: int = 10, margin: int = 5) -> np.ndarray:
    """raw_v1 copy of the training script's crop_empty_borders() (margin 5, not 4)."""
    mask = img > threshold
    if not np.any(mask):
        return img
    y_idx, x_idx = np.where(mask)
    h, w = img.shape[:2]
    ymin = max(0, y_idx.min() - margin)
    ymax = min(h, y_idx.max() + margin + 1)
    xmin = max(0, x_idx.min() - margin)
    xmax = min(w, x_idx.max() + margin + 1)
    return img[ymin:ymax, xmin:xmax]


def compute_slice_position(dcm: pydicom.Dataset) -> float:
    ori = getattr(dcm, "ImageOrientationPatient", [1, 0, 0, 0, 1, 0])
    pos = getattr(dcm, "ImagePositionPatient", [0, 0, 0])
    f = np.array(ori[:3], dtype=float)
    c = np.array(ori[3:], dtype=float)
    normal = np.cross(f, c)
    norm = np.linalg.norm(normal)
    normal = normal / norm if norm > 1e-6 else np.array([0.0, 0.0, 1.0])
    return float(np.dot(normal, np.array(pos, dtype=float)))


def load_series_raw_v1(dicom_files: List[str], target_size=(256, 256), num_slices: int = 16) -> np.ndarray:
    """Fast multithreaded header-first raw_v1 loader: sorts headers via stop_before_pixels=True
    and decodes pixels ONLY for the 16 sampled indices and their immediate 2.5D neighbors."""
    if not dicom_files:
        return np.zeros((num_slices, 3, target_size[0], target_size[1]), dtype=np.float32)

    def _hdr_pos(fpath):
        _, spos, ok = _get_cached_hdr(fpath)
        return (spos, fpath) if ok else None

    hdrs = [h for h in _IO_POOL.map(_hdr_pos, dicom_files) if h is not None]

    if not hdrs:
        return np.zeros((num_slices, 3, target_size[0], target_size[1]), dtype=np.float32)

    hdrs.sort(key=lambda x: x[0])
    n_slices = len(hdrs)
    indices = np.linspace(0, n_slices - 1, num_slices).round().astype(int)
    needed_idx = sorted({max(0, i - 1) for i in indices} | set(indices) | {min(n_slices - 1, i + 1) for i in indices})

    def _decode_idx(idx):
        fpath = hdrs[idx][1]
        try:
            dcm = pydicom.dcmread(fpath)
            photometric = getattr(dcm, "PhotometricInterpretation", "MONOCHROME2")
            wc = getattr(dcm, "WindowCenter", None)
            ww = getattr(dcm, "WindowWidth", None)
            pixels = dcm.pixel_array.astype(np.float32)
            slope = float(getattr(dcm, "RescaleSlope", 1.0))
            intercept = float(getattr(dcm, "RescaleIntercept", 0.0))
            if slope != 1.0 or intercept != 0.0:
                pixels = pixels * slope + intercept
            img = apply_windowing_raw(pixels, center=wc, width=ww, photometric=photometric)
            img = crop_empty_borders_raw(img)
            img = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
            return idx, img
        except Exception:
            return idx, np.zeros(target_size, dtype=np.uint8)

    decoded = dict(_IO_POOL.map(_decode_idx, needed_idx))

    slabs = []
    for idx in indices:
        z_prev = decoded[max(0, idx - 1)]
        z_curr = decoded[idx]
        z_next = decoded[min(n_slices - 1, idx + 1)]
        slab = np.stack([z_prev, z_curr, z_next], axis=0).astype(np.float32) / 255.0
        slab = (slab - IMAGENET_MEAN) / IMAGENET_STD
        slabs.append(slab)

    return np.stack(slabs, axis=0).astype(np.float32)


def select_series_uid(study_series: pd.DataFrame, plane_col: Optional[str], plane: str, pipeline: str) -> Optional[str]:
    """Pick the series for a plane the same way the model's training pipeline did."""
    if study_series.empty or plane_col is None:
        return None
    values = study_series[plane_col].astype(str)
    if pipeline in ("cache_v1", "cache_v1_384"):
        match = study_series[values.str.capitalize() == plane]
        if not match.empty:
            if "Fluid_Sensitive" in match.columns or "Fat_Suppression" in match.columns:
                fluid = match.get("Fluid_Sensitive", pd.Series(0, index=match.index)).fillna(0).astype(int)
                fat = match.get("Fat_Suppression", pd.Series(0, index=match.index)).fillna(0).astype(int)
                match = match.assign(_prio=(fluid * 2) + fat).sort_values("_prio", ascending=False)
            return str(match.iloc[0]["SeriesInstanceUID"])
    # raw_v1 rule, also the fallback for free-text plane columns like SeriesDescription
    match = study_series[values.str.lower().str.contains(plane.lower())]
    return str(match.iloc[0]["SeriesInstanceUID"]) if not match.empty else None


def load_plane(dicom_files: List[str], pipeline: str, target_size=(256, 256), num_slices: int = 16) -> torch.Tensor:
    """DICOM files of one series -> (K, 3, H, W) tensor, preprocessed like the given pipeline."""
    if pipeline == "cache_v1":
        slabs = cached_plane_to_slabs(sample_plane_like_cache(dicom_files, num_slices=num_slices, target_size=target_size))
    elif pipeline == "cache_v1_384":
        slabs = cached_plane_to_slabs(sample_plane_like_cache(dicom_files, num_slices=num_slices, target_size=(384, 384)))
    elif pipeline == "raw_v1":
        slabs = load_series_raw_v1(dicom_files, target_size=target_size, num_slices=num_slices)
    else:
        raise ValueError(f"Unknown preprocessing pipeline: {pipeline}")
    return torch.tensor(slabs, dtype=torch.float32)


def checkpoint_pipeline(state: dict, backbone: str) -> str:
    """Which preprocessing a checkpoint was trained with.

    New checkpoints record it. Older ones don't: the resnet34 run predates the slice cache
    (raw_v1), and the ConvNeXt run was the first trained on the cache (cache_v1).
    384px CoAtNet checkpoints trained on the 384px N4 cache map to 'cache_v1_384'.
    """
    pipe = state.get("preprocessing") if isinstance(state, dict) else None
    if pipe is None:
        pipe = "raw_v1" if backbone.startswith("resnet") else "cache_v1"
        print(f"    (checkpoint does not record its preprocessing; assuming '{pipe}' for backbone '{backbone}')")
    cfg = state.get("config", {}) if (isinstance(state, dict) and isinstance(state.get("config"), dict)) else {}
    ts = cfg.get("target_size")
    is_384 = (
        (isinstance(ts, (list, tuple)) and len(ts) == 2 and int(ts[0]) == 384)
        or ("384" in backbone)
        or (cfg.get("timm_img_size") == 384)
    )
    if pipe == "cache_v1" and is_384:
        pipe = "cache_v1_384"
    if pipe not in PIPELINES:
        raise ValueError(f"Checkpoint uses unknown preprocessing '{pipe}'")
    return pipe


# ==============================================================================
# MODEL DEFINITION (SELF-CONTAINED)
# ==============================================================================

class GatedAttentionPool(nn.Module):
    def __init__(self, in_features: int, hidden_dim: int = 128):
        super().__init__()
        self.v_proj = nn.Linear(in_features, hidden_dim)
        self.u_proj = nn.Linear(in_features, hidden_dim)
        self.w_proj = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        v = torch.tanh(self.v_proj(x))
        u = torch.sigmoid(self.u_proj(x))
        attn_scores = self.w_proj(v * u)
        attn_weights = F.softmax(attn_scores, dim=1)
        pooled = torch.sum(x * attn_weights, dim=1)
        return pooled


class KneeAnatomicalMoEClassifier(nn.Module):
    """Multi-Planar 2.5D Knee Model with Anatomical Plane-Specific Expert Routing."""
    def __init__(
        self,
        backbone_name: str = "resnet34",
        pretrained: bool = False,
        num_classes: int = 12,
        dropout: float = 0.2,
        timm_img_size: Optional[int] = None,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.backbone_name = backbone_name

        # Vision Backbone (ResNet, ConvNeXt, EfficientNet, CoAtNet)
        try:
            if backbone_name == "resnet34":
                model = tv_models.resnet34(weights=None)
                self.feat_dim = model.fc.in_features
                model.fc = nn.Identity()
                self.encoder = model
            elif backbone_name == "resnet50":
                model = tv_models.resnet50(weights=None)
                self.feat_dim = model.fc.in_features
                model.fc = nn.Identity()
                self.encoder = model
            elif backbone_name in ["convnext_tiny", "convnext_small"]:
                factory = tv_models.convnext_small if backbone_name == "convnext_small" else tv_models.convnext_tiny
                model = factory(weights=None)
                self.feat_dim = model.classifier[2].in_features
                model.classifier[2] = nn.Identity()
                self.encoder = model
            elif backbone_name in ["efficientnet_v2_s", "effnet"]:
                model = tv_models.efficientnet_v2_s(weights=None)
                self.feat_dim = model.classifier[1].in_features
                model.classifier[1] = nn.Identity()
                self.encoder = model
            elif backbone_name.startswith("coatnet"):
                import timm
                kw = dict(pretrained=False, num_classes=0, in_chans=3)
                if timm_img_size is not None:
                    kw["img_size"] = int(timm_img_size)
                elif "224" in backbone_name:
                    kw["img_size"] = 384
                model = timm.create_model(backbone_name, **kw)
                self.feat_dim = model.num_features
                self.encoder = model
            else:
                model = getattr(tv_models, backbone_name)(weights=None)
                self.feat_dim = model.fc.in_features
                model.fc = nn.Identity()
                self.encoder = model
        except Exception as e:
            # No silent fallback: a substitute backbone would run with random weights.
            raise RuntimeError(f"Could not build backbone '{backbone_name}': {e}") from e

        self.sag_pool = GatedAttentionPool(self.feat_dim)
        self.cor_pool = GatedAttentionPool(self.feat_dim)
        self.ax_pool = GatedAttentionPool(self.feat_dim)

        combined_dim = self.feat_dim * 3

        self.sag_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feat_dim + combined_dim, 256),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 3)
        )

        self.cor_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feat_dim + combined_dim, 128),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

        self.ax_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feat_dim + combined_dim, 256),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 3)
        )

        self.joint_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(combined_dim, 256),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 5)
        )

    def _encode_plane(self, x: torch.Tensor, pool_module: GatedAttentionPool) -> torch.Tensor:
        B, K, C, H, W = x.shape
        x_flat = x.view(B * K, C, H, W)
        feats = self.encoder(x_flat)
        if isinstance(feats, torch.Tensor) and feats.dim() > 2:
            feats = feats.flatten(1)
        feats = feats.view(B, K, self.feat_dim)
        return pool_module(feats)

    def forward(self, sag: torch.Tensor, cor: torch.Tensor, ax: torch.Tensor) -> torch.Tensor:
        B, K, C, H, W = sag.shape
        x_all = torch.cat([sag, cor, ax], dim=1).reshape(B * 3 * K, C, H, W)
        feats = self.encoder(x_all)
        if isinstance(feats, torch.Tensor) and feats.dim() > 2:
            feats = feats.flatten(1)
        feats = feats.view(B, 3 * K, self.feat_dim)
        h_sag = self.sag_pool(feats[:, :K])
        h_cor = self.cor_pool(feats[:, K : 2 * K])
        h_ax = self.ax_pool(feats[:, 2 * K :])

        combined = torch.cat([h_sag, h_cor, h_ax], dim=1)

        out_sag = self.sag_head(torch.cat([h_sag, combined], dim=1))
        out_cor = self.cor_head(torch.cat([h_cor, combined], dim=1))
        out_ax = self.ax_head(torch.cat([h_ax, combined], dim=1))
        out_joint = self.joint_head(combined)

        logits = torch.stack([
            out_sag[:, 0],     # ACL
            out_cor[:, 0],     # MCL
            out_sag[:, 1],     # Medial Meniscus
            out_sag[:, 2],     # Lateral Meniscus
            out_joint[:, 0],   # Medial OA
            out_joint[:, 1],   # Lateral OA
            out_ax[:, 0],      # PF OA
            out_ax[:, 1],      # Effusion
            out_ax[:, 2],      # Synovitis
            out_joint[:, 2],   # Baker's
            out_joint[:, 3],   # Contusion
            out_joint[:, 4],   # Fracture
        ], dim=1)

        return logits


class LegacyKneeClassifier(nn.Module):
    """Fallback legacy classifier for older checkpoints."""
    def __init__(self, feat_dim: int = 512, num_classes: int = 12, dropout: float = 0.2):
        super().__init__()
        self.feat_dim = feat_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.SiLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.SiLU(),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.SiLU(),
            nn.Conv2d(256, feat_dim, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(feat_dim),
            nn.SiLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.sag_pool = GatedAttentionPool(feat_dim)
        self.cor_pool = GatedAttentionPool(feat_dim)
        self.ax_pool = GatedAttentionPool(feat_dim)

        combined_dim = feat_dim * 3
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(combined_dim, 512),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def _encode_plane(self, x: torch.Tensor, pool_module: GatedAttentionPool) -> torch.Tensor:
        B, K, C, H, W = x.shape
        x_flat = x.view(B * K, C, H, W)
        feats = self.encoder(x_flat).flatten(1)
        feats = feats.view(B, K, self.feat_dim)
        return pool_module(feats)

    def forward(self, sag: torch.Tensor, cor: torch.Tensor, ax: torch.Tensor) -> torch.Tensor:
        h_sag = self._encode_plane(sag, self.sag_pool)
        h_cor = self._encode_plane(cor, self.cor_pool)
        h_ax = self._encode_plane(ax, self.ax_pool)
        combined = torch.cat([h_sag, h_cor, h_ax], dim=1)
        return self.head(combined)

KneeAbnormalityClassifier = KneeAnatomicalMoEClassifier



def find_series_files(study_uid: str, study_series: pd.DataFrame, plane_col: Optional[str], plane: str,
                      pipeline: str, test_series_dir: Optional[str]) -> List[str]:
    series_files: List[str] = []
    series_uid = select_series_uid(study_series, plane_col, plane, pipeline)
    if series_uid is not None:
        # Look in standard Kaggle hierarchy
        if test_series_dir and os.path.isdir(test_series_dir):
            series_files = glob.glob(os.path.join(test_series_dir, study_uid, series_uid, "*.dcm"))
            if not series_files:
                # Try flat series subfolder
                series_files = glob.glob(os.path.join(test_series_dir, series_uid, "*.dcm"))
            if not series_files:
                # Try flat study subfolder
                series_files = glob.glob(os.path.join(test_series_dir, study_uid, "*.dcm"))
            if not series_files:
                # Try direct flat DICOM folder (e.g. sample_dicom/*.dcm)
                series_files = glob.glob(os.path.join(test_series_dir, "*.dcm"))
    elif test_series_dir and os.path.isdir(test_series_dir):
        # No plane column — distribute the study's series across planes
        study_dir = os.path.join(test_series_dir, study_uid)
        if os.path.isdir(study_dir):
            series_dirs = [d for d in os.listdir(study_dir) if os.path.isdir(os.path.join(study_dir, d))]
            plane_idx = ["Sagittal", "Coronal", "Axial"].index(plane)
            if plane_idx < len(series_dirs):
                series_files = glob.glob(os.path.join(study_dir, series_dirs[plane_idx], "*.dcm"))
        if not series_files:
            series_files = glob.glob(os.path.join(test_series_dir, "*.dcm"))
    return series_files


# ==============================================================================
# MAIN INFERENCE ENGINE
# ==============================================================================

import multiprocessing as _s2_mp
_S2_SERIES_BY_STUDY = {}
_S2_PLANE_COL = None
_S2_TSDIR = None
def _s2_worker_init():
    global _IO_POOL
    _IO_POOL = _PartCThreadPool(max_workers=12)
def _s2_worker_fn(args):
    idx, study_uid = args
    study_series = _S2_SERIES_BY_STUDY.get(study_uid, pd.DataFrame())
    u8_planes = {}
    for plane in ("Sagittal", "Coronal", "Axial"):
        series_files = find_series_files(study_uid, study_series, _S2_PLANE_COL, plane, "cache_v1_384", _S2_TSDIR)
        u8_planes[plane] = sample_plane_like_cache(series_files, num_slices=16, target_size=(384, 384))
    if len(_HDR_CACHE) > 15000:
        _HDR_CACHE.clear()
    return idx, study_uid, u8_planes

def run_submission_inference(output_path: str = "submission.csv") -> pd.DataFrame:
    """Run full inference loop and generate submission.csv."""
    test_csv, test_series_csv, test_series_dir, ckpt_files = get_data_paths()
    ckpt_files = sorted(set(ckpt_files))
    if CHECKPOINT_FILTER:
        skipped = [c for c in ckpt_files if not any(f in c for f in CHECKPOINT_FILTER)]
        ckpt_files = [c for c in ckpt_files if any(f in c for f in CHECKPOINT_FILTER)]
        print(f"CHECKPOINT_FILTER {CHECKPOINT_FILTER}: using {len(ckpt_files)}, skipping {len(skipped)}")
        for c in skipped:
            print(f"  skipped: {c}")
        if not ckpt_files:
            raise RuntimeError(f"CHECKPOINT_FILTER {CHECKPOINT_FILTER} matched no attached checkpoint.")

    print(f"Reading test data from: {test_csv}")
    if test_csv is None or not os.path.exists(test_csv):
        print("WARNING: test.csv not found anywhere! Generating fallback submission with priors.")
        # List what we can see for debugging
        if os.path.exists("/kaggle/input"):
            print("Top-level /kaggle/input contents:")
            try:
                for entry in os.listdir("/kaggle/input"):
                    print(f"  {entry}/")
                    entry_path = os.path.join("/kaggle/input", entry)
                    if os.path.isdir(entry_path):
                        for sub in os.listdir(entry_path)[:10]:  # limit to first 10
                            print(f"    {sub}")
            except OSError as e:
                print(f"  Error listing: {e}")
        # Can't proceed without any test data — create empty submission
        sub_df = pd.DataFrame(columns=["StudyInstanceUID"] + TARGET_COLS)
        sub_df.to_csv(output_path, index=False)
        print(f"Created empty submission file: {output_path}")
        return sub_df

    test_df = pd.read_csv(test_csv)
    test_series_df = pd.read_csv(test_series_csv) if (test_series_csv and os.path.exists(test_series_csv)) else pd.DataFrame()

    print(f"Loaded {len(test_df)} test studies.")
    if not test_series_df.empty:
        print(f"Loaded {len(test_series_df)} series entries. Columns: {list(test_series_df.columns)}")

    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Inference Device: {device}")

    # Detect plane column and prepare study metadata FIRST so we can start the 4-process
    # multiprocessing DICOM pool BEFORE loading the 6 GPU checkpoints!
    plane_col = None
    if not test_series_df.empty:
        for col_candidate in ["Anatomical_Plane", "anatomical_plane", "Plane", "SeriesDescription"]:
            if col_candidate in test_series_df.columns:
                plane_col = col_candidate
                print(f"Using plane column: {plane_col}")
                break

    series_by_study = (
        {str(uid): grp for uid, grp in test_series_df.groupby("StudyInstanceUID")}
        if not test_series_df.empty and "StudyInstanceUID" in test_series_df.columns
        else {}
    )
    study_uids_list = [str(r["StudyInstanceUID"]) for _, r in test_df.iterrows()]

    global _S2_SERIES_BY_STUDY, _S2_PLANE_COL, _S2_TSDIR
    _S2_SERIES_BY_STUDY = series_by_study
    _S2_PLANE_COL = plane_col
    _S2_TSDIR = test_series_dir
    _s2_pool = _s2_mp.get_context("fork").Pool(4, initializer=_s2_worker_init)
    _s2_iter = _s2_pool.imap(_s2_worker_fn, enumerate(study_uids_list), chunksize=2)

    # Check for model checkpoints and load all 6 in parallel onto balanced dual GPUs
    models = []
    if ckpt_files:
        print(f"Found {len(ckpt_files)} model checkpoints. Loading ensemble onto balanced dual GPUs...")
        def _load_one_ckpt(ckpt_path):
            state = torch.load(ckpt_path, map_location="cpu")
            sd = state["model_state_dict"] if (isinstance(state, dict) and "model_state_dict" in state) else state
            arch = state.get("architecture", "") if isinstance(state, dict) else ""
            bb = state.get("backbone", "resnet34") if isinstance(state, dict) else "resnet34"
            cfg = state.get("config", {}) if (isinstance(state, dict) and isinstance(state.get("config"), dict)) else {}
            timm_img_size = cfg.get("timm_img_size")
            is_moe = any("sag_head" in k for k in sd.keys()) or arch == "KneeAnatomicalMoEClassifier"
            if is_moe:
                model = KneeAnatomicalMoEClassifier(backbone_name=bb, pretrained=False, timm_img_size=timm_img_size)
            else:
                model = LegacyKneeClassifier()
            model.load_state_dict(sd, strict=True)
            model.eval()
            pipeline = checkpoint_pipeline(state, bb)
            ckpt_fold = state.get("fold") if isinstance(state, dict) else None
            if ckpt_fold is None:
                m_f = re.search(r"fold_?(\d+)", os.path.basename(ckpt_path))
                ckpt_fold = int(m_f.group(1)) if m_f else None
            # 50/50 Dual-GPU FLOP Sharding:
            # cuda:0 -> C384 Fold 0 (CoAtNet-2 @ 384) + C384 Fold 1 (CoAtNet-1 @ 384) [~155ms]
            # cuda:1 -> C384 Fold 2 (CoAtNet-2 @ 384) + 3x N4 (ConvNeXt-Small @ 256)  [~155ms]
            is_gpu1 = (pipeline in ("cache_v1", "raw_v1") or (pipeline == "cache_v1_384" and str(ckpt_fold) == "2"))
            target_dev = torch.device("cuda:1") if (is_gpu1 and torch.cuda.device_count() >= 2) else device
            model = model.to(target_dev)
            print(f"  Loaded model from: {ckpt_path} -> {target_dev} (MoE: {is_moe}, Backbone: {bb}, Preprocessing: {pipeline}, Fold: {ckpt_fold})")
            return (model, pipeline, ckpt_fold)

        try:
            models = list(_IO_POOL.map(_load_one_ckpt, ckpt_files))
        except Exception as e:
            _s2_pool.terminate()
            raise RuntimeError(f"Failed to load checkpoints: {e}") from e

    if not models and os.path.exists("/kaggle/input"):
        _s2_pool.terminate()
        raise RuntimeError("No model checkpoints found under /kaggle/input. Attach the trained model before submitting.")

    submission_rows = []
    pipelines_in_use = sorted({m[1] for m in models}) or ["cache_v1"]

    for idx, study_uid, u8_planes in _s2_iter:
        if idx < 3:
            print(f"  Study {study_uid[:15]}... | mp_fork_pool_u8 | Loaded 3x(16, 384, 384) uint8")
        base_u8 = {k: torch.from_numpy(v) for k, v in u8_planes.items()}
        plane_tensors = {p: base_u8 for p in pipelines_in_use}
        pred_dict = {"StudyInstanceUID": study_uid}

        # 3-Fold (2,610 Silver OOF studies) OOF-Anchored Z-Logit Calibration + Adaptive Weights
        # Lifts 2,610-study Silver OOF to 0.8665 and 58-study Out-of-Sample Gold Benchmark to 0.8561 (+0.0156 over N4)
        C384_ADAPTIVE_WEIGHTS = {
            "ACL": 0.230,
            "MCL": 0.270,
            "Medial Meniscus": 0.290,
            "Lateral Meniscus": 0.290,
            "Medial OA": 0.220,
            "Lateral OA": 0.180,
            "PF OA": 0.220,
            "Effusion": 0.500,
            "Synovitis": 0.350,
            "Baker's": 0.380,
            "Contusion": 0.270,
            "Fracture": 0.310,
        }
        OOF_ZLOGIT_STATS = {
                    "n4": {
                                "0": {
                                            "ACL": [
                                                        -1.7938,
                                                        2.07134
                                            ],
                                            "MCL": [
                                                        -2.15473,
                                                        1.39975
                                            ],
                                            "Medial Meniscus": [
                                                        -0.02618,
                                                        2.67454
                                            ],
                                            "Lateral Meniscus": [
                                                        -1.58914,
                                                        1.85337
                                            ],
                                            "Medial OA": [
                                                        -1.10909,
                                                        2.00486
                                            ],
                                            "Lateral OA": [
                                                        -1.83539,
                                                        1.28232
                                            ],
                                            "PF OA": [
                                                        -0.958,
                                                        1.58485
                                            ],
                                            "Effusion": [
                                                        0.98501,
                                                        2.30042
                                            ],
                                            "Synovitis": [
                                                        -0.92933,
                                                        1.3895
                                            ],
                                            "Baker's": [
                                                        -1.16773,
                                                        2.53981
                                            ],
                                            "Contusion": [
                                                        -0.99397,
                                                        1.93469
                                            ],
                                            "Fracture": [
                                                        -2.46582,
                                                        0.98994
                                            ]
                                },
                                "1": {
                                            "ACL": [
                                                        -1.79655,
                                                        1.91032
                                            ],
                                            "MCL": [
                                                        -2.06606,
                                                        1.31552
                                            ],
                                            "Medial Meniscus": [
                                                        0.39188,
                                                        2.64627
                                            ],
                                            "Lateral Meniscus": [
                                                        -1.59308,
                                                        1.57453
                                            ],
                                            "Medial OA": [
                                                        -1.24129,
                                                        1.89935
                                            ],
                                            "Lateral OA": [
                                                        -1.99832,
                                                        1.1452
                                            ],
                                            "PF OA": [
                                                        -1.16561,
                                                        1.52212
                                            ],
                                            "Effusion": [
                                                        0.85379,
                                                        2.41984
                                            ],
                                            "Synovitis": [
                                                        -0.90554,
                                                        1.39356
                                            ],
                                            "Baker's": [
                                                        -1.40945,
                                                        2.30303
                                            ],
                                            "Contusion": [
                                                        -1.20082,
                                                        1.80139
                                            ],
                                            "Fracture": [
                                                        -2.42407,
                                                        1.02522
                                            ]
                                },
                                "2": {
                                            "ACL": [
                                                        -2.00793,
                                                        1.915
                                            ],
                                            "MCL": [
                                                        -2.32746,
                                                        1.32444
                                            ],
                                            "Medial Meniscus": [
                                                        0.21541,
                                                        2.66373
                                            ],
                                            "Lateral Meniscus": [
                                                        -1.47953,
                                                        1.79538
                                            ],
                                            "Medial OA": [
                                                        -1.16182,
                                                        2.0183
                                            ],
                                            "Lateral OA": [
                                                        -1.93131,
                                                        1.22867
                                            ],
                                            "PF OA": [
                                                        -1.16446,
                                                        1.53026
                                            ],
                                            "Effusion": [
                                                        1.06338,
                                                        2.2962
                                            ],
                                            "Synovitis": [
                                                        -0.84133,
                                                        1.39412
                                            ],
                                            "Baker's": [
                                                        -1.09215,
                                                        2.53914
                                            ],
                                            "Contusion": [
                                                        -1.06193,
                                                        1.8333
                                            ],
                                            "Fracture": [
                                                        -2.44681,
                                                        0.95287
                                            ]
                                }
                    },
                    "c384": {
                                "0": {
                                            "ACL": [
                                                        -1.57795,
                                                        1.08777
                                            ],
                                            "MCL": [
                                                        -1.85532,
                                                        0.84493
                                            ],
                                            "Medial Meniscus": [
                                                        0.25548,
                                                        1.77119
                                            ],
                                            "Lateral Meniscus": [
                                                        -1.41753,
                                                        1.20038
                                            ],
                                            "Medial OA": [
                                                        -1.2835,
                                                        1.4106
                                            ],
                                            "Lateral OA": [
                                                        -1.90778,
                                                        0.8405
                                            ],
                                            "PF OA": [
                                                        -1.16651,
                                                        1.15357
                                            ],
                                            "Effusion": [
                                                        0.98819,
                                                        1.78023
                                            ],
                                            "Synovitis": [
                                                        -0.91018,
                                                        1.17204
                                            ],
                                            "Baker's": [
                                                        -1.41221,
                                                        2.02268
                                            ],
                                            "Contusion": [
                                                        -0.9679,
                                                        1.18356
                                            ],
                                            "Fracture": [
                                                        -2.19061,
                                                        0.63896
                                            ]
                                },
                                "1": {
                                            "ACL": [
                                                        -1.26711,
                                                        1.21898
                                            ],
                                            "MCL": [
                                                        -1.52635,
                                                        0.87659
                                            ],
                                            "Medial Meniscus": [
                                                        0.54775,
                                                        1.4837
                                            ],
                                            "Lateral Meniscus": [
                                                        -0.98085,
                                                        1.07125
                                            ],
                                            "Medial OA": [
                                                        -0.92322,
                                                        1.25914
                                            ],
                                            "Lateral OA": [
                                                        -1.70659,
                                                        0.83059
                                            ],
                                            "PF OA": [
                                                        -0.85812,
                                                        1.02504
                                            ],
                                            "Effusion": [
                                                        0.58761,
                                                        1.56557
                                            ],
                                            "Synovitis": [
                                                        -0.93816,
                                                        1.04135
                                            ],
                                            "Baker's": [
                                                        -1.00625,
                                                        1.55821
                                            ],
                                            "Contusion": [
                                                        -0.2907,
                                                        1.55658
                                            ],
                                            "Fracture": [
                                                        -1.66078,
                                                        1.09701
                                            ]
                                },
                                "2": {
                                            "ACL": [
                                                        -1.38427,
                                                        1.24273
                                            ],
                                            "MCL": [
                                                        -1.96517,
                                                        0.76715
                                            ],
                                            "Medial Meniscus": [
                                                        0.52747,
                                                        2.01158
                                            ],
                                            "Lateral Meniscus": [
                                                        -1.24149,
                                                        1.30375
                                            ],
                                            "Medial OA": [
                                                        -1.02715,
                                                        1.62696
                                            ],
                                            "Lateral OA": [
                                                        -1.7013,
                                                        0.94094
                                            ],
                                            "PF OA": [
                                                        -0.86659,
                                                        1.20024
                                            ],
                                            "Effusion": [
                                                        0.45869,
                                                        1.83928
                                            ],
                                            "Synovitis": [
                                                        -1.01878,
                                                        1.17997
                                            ],
                                            "Baker's": [
                                                        -1.23132,
                                                        2.09861
                                            ],
                                            "Contusion": [
                                                        -1.15817,
                                                        1.32209
                                            ],
                                            "Fracture": [
                                                        -2.37418,
                                                        0.72254
                                            ]
                                }
                    }
        }

        def _make_zoom_tta(x: torch.Tensor, crop_ratio: float = 0.96) -> torch.Tensor:
            # Pure in-VRAM 1.04x center-crop zoom TTA across all slices (1, K, 3, H, W) -> (1, K, 3, H, W)
            b, k, c, h, w = x.shape
            ch, cw = int(round(h * crop_ratio)), int(round(w * crop_ratio))
            y0, x0 = (h - ch) // 2, (w - cw) // 2
            cropped = x[:, :, :, y0:y0 + ch, x0:x0 + cw].reshape(b * k, c, ch, cw)
            zoomed = F.interpolate(cropped, size=(h, w), mode="bilinear", align_corners=False)
            return zoomed.reshape(b, k, c, h, w)

        # Run Model Inference concurrently across cuda:0 (C384) and cuda:1 (N4) in FP16
        if models:
            convnext_family = []
            coatnet_family = []
            other_family = []

            def _eval_device_group(dev_models):
                out_list = []
                if not dev_models:
                    return out_list
                m_dev = next(dev_models[0][0].parameters()).device
                gpu_pt_cache = {}
                with torch.no_grad():
                    for model_item in dev_models:
                        model, pipeline = model_item[0], model_item[1]
                        ckpt_fold = model_item[2] if len(model_item) > 2 else None
                        if pipeline not in gpu_pt_cache:
                            if "_base384" not in gpu_pt_cache:
                                pt = plane_tensors[pipeline]
                                _ip = [max(0, i - 1) for i in range(16)]
                                _in = [min(15, i + 1) for i in range(16)]
                                _gm = torch.tensor([0.485, 0.456, 0.406], device=m_dev, dtype=torch.float32).view(1, 3, 1, 1)
                                _gsd = torch.tensor([0.229, 0.224, 0.225], device=m_dev, dtype=torch.float32).view(1, 3, 1, 1)
                                def _u8_to_384(u8_t):
                                    g_u8 = u8_t.to(m_dev, non_blocking=True)
                                    sl = torch.stack([g_u8[_ip], g_u8, g_u8[_in]], dim=1).to(torch.float32).mul_(1.0 / 255.0)
                                    return sl.sub_(_gm).div_(_gsd).unsqueeze(0)
                                gpu_pt_cache["_base384"] = (_u8_to_384(pt["Sagittal"]), _u8_to_384(pt["Coronal"]), _u8_to_384(pt["Axial"]))
                            gs, gc_, ga = gpu_pt_cache["_base384"]
                            if pipeline in ("cache_v1", "raw_v1"):
                                gs = F.interpolate(gs.view(-1, 3, 384, 384), size=(256, 256), mode="bilinear", align_corners=False).view(1, -1, 3, 256, 256)
                                gc_ = F.interpolate(gc_.view(-1, 3, 384, 384), size=(256, 256), mode="bilinear", align_corners=False).view(1, -1, 3, 256, 256)
                                ga = F.interpolate(ga.view(-1, 3, 384, 384), size=(256, 256), mode="bilinear", align_corners=False).view(1, -1, 3, 256, 256)
                            gpu_pt_cache[pipeline] = (gs, gc_, ga)
                        gsag, gcor, gax = gpu_pt_cache[pipeline]
                        with torch.amp.autocast('cuda', enabled=(m_dev.type == "cuda")):
                            logits_1 = model(gsag, gcor, gax).float()
                        out_list.append((model, ckpt_fold, logits_1.cpu().numpy()[0]))
                return out_list

            dev_groups = {}
            for m_item in models:
                d_str = str(next(m_item[0].parameters()).device)
                dev_groups.setdefault(d_str, []).append(m_item)

            if len(dev_groups) >= 2:
                group_results = list(_GPU_POOL.map(_eval_device_group, dev_groups.values()))
                all_evals = [item for grp in group_results for item in grp]
            else:
                all_evals = _eval_device_group(models)

            for model, ckpt_fold, raw_logits in all_evals:
                if True:
                    bb = getattr(model, "backbone_name", "")
                    fam_key = "c384" if bb.startswith("coatnet") else ("n4" if bb in ["convnext_small", "convnext_tiny"] else None)
                    fold_key = str(ckpt_fold) if ckpt_fold in (0, 1, 2, "0", "1", "2") else None

                    if fam_key and fold_key and fold_key in OOF_ZLOGIT_STATS.get(fam_key, {}):
                        f_stats = OOF_ZLOGIT_STATS[fam_key][fold_key]
                        z_vec = np.zeros(len(TARGET_COLS), dtype=np.float32)
                        for t_idx, target in enumerate(TARGET_COLS):
                            mu_t, sd_t = f_stats[target]
                            clipped_l = float(np.clip(raw_logits[t_idx], -11.5, 11.5))
                            z_vec[t_idx] = (clipped_l - mu_t) / sd_t
                        if fam_key == "c384":
                            coatnet_family.append(z_vec)
                        else:
                            convnext_family.append(z_vec)
                    else:
                        probs = 1.0 / (1.0 + np.exp(-np.clip(raw_logits, -11.5, 11.5)))
                        other_family.append(probs)

            if convnext_family and coatnet_family:
                cn_avg = np.mean(convnext_family, axis=0)
                ca_avg = np.mean(coatnet_family, axis=0)
                avg_probs = np.zeros(len(TARGET_COLS), dtype=np.float32)
                for t_idx, target in enumerate(TARGET_COLS):
                    w_ca = float(C384_ADAPTIVE_WEIGHTS.get(target, 0.25))
                    z_blend = (1.0 - w_ca) * cn_avg[t_idx] + w_ca * ca_avg[t_idx]
                    avg_probs[t_idx] = float(1.0 / (1.0 + np.exp(-np.clip(z_blend, -11.5, 11.5))))
            elif convnext_family or coatnet_family:
                z_all = convnext_family + coatnet_family
                z_avg = np.mean(z_all, axis=0)
                avg_probs = 1.0 / (1.0 + np.exp(-np.clip(z_avg, -11.5, 11.5)))
            else:
                avg_probs = np.mean(other_family, axis=0)

            for t_idx, target in enumerate(TARGET_COLS):
                pred_dict[target] = float(np.clip(avg_probs[t_idx], 0.001, 0.999))
        else:
            # Fallback to calibrated dataset priors
            for target in TARGET_COLS:
                pred_dict[target] = DEFAULT_PRIORS[target]

        submission_rows.append(pred_dict)

    _s2_pool.close(); _s2_pool.join()
    sub_df = pd.DataFrame(submission_rows)
    # Ensure columns match exact competition order
    final_cols = ["StudyInstanceUID"] + TARGET_COLS
    sub_df = sub_df[final_cols]

    # Sanity checks to prevent constant 0.500 fallback
    if len(sub_df) > 1:
        stds = [sub_df[t].std() for t in TARGET_COLS]
        avg_std = float(np.mean(stds))
        print(f"Submission sanity check: Average per-target std dev = {avg_std:.6f}")
        if avg_std < 1e-4:
            print("[!] CRITICAL WARNING: Predictions appear to be identical constants across test studies!")
            print("    Check that model checkpoints and DICOM files are properly loaded.")
        else:
            print("[OK] Predictions vary dynamically across test cases.")

    sub_df.to_csv(output_path, index=False)
    print(f"\nSuccessfully generated submission file: {output_path}")
    print(f"Shape: {sub_df.shape}")
    print(sub_df.head())

    return sub_df




import shutil as _sh, time as _c_time
_sh.copy('submission.csv', 'public_submission_backup.csv')
try:
    if _c_time.time() - globals().get('NB_T0', _c_time.time()) > 8.5 * 3600:
        raise RuntimeError('Time budget (>8.5h) reached before Part C; keeping 0.9586 stack submission')
    run_submission_inference("our_submission.csv")
    print('Blending Apex Foundation with Creaky E24 Multi-Resolution MoE (N4 256px ConvNeXt + C384 384px CoAtNet)!')
    import pandas as pd
    import gc
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    try:
        their_sub = pd.read_csv('submission.csv')
        our_sub = pd.read_csv('our_submission.csv')
        their_sub['StudyInstanceUID'] = their_sub['StudyInstanceUID'].astype(str)
        our_sub['StudyInstanceUID'] = our_sub['StudyInstanceUID'].astype(str)
        our_sub = our_sub.set_index('StudyInstanceUID').reindex(their_sub['StudyInstanceUID']).reset_index()
        assert not our_sub.isna().any().any(), 'study ID mismatch between submissions'

        TARGET_COLS = [c for c in their_sub.columns if c != 'StudyInstanceUID']
        our_ranks = our_sub[TARGET_COLS].rank(method="average", pct=True)

        # Micro-injection weights calibrated to 6-Model Z-Logit Gold Benchmark (0.8568 Gold Macro AUC / 0.8676 Silver OOF)
        # on top of the 0.9586 4-family stack (Raptor-384 + Raptor-224 + R6 Flip-TTA + 2.5D MIL):
        # Elite tier (>0.92 Gold AUC): Medial OA (0.9845), Baker's (0.9801), Fracture (0.9208) -> 0.040-0.045
        # Strong tier (0.85-0.90 Gold AUC): Contusion (0.8853), ACL (0.8848), MCL (0.8753), Effusion (0.8821 OOF, +0.0160 lift) -> 0.035
        # Mid/Conservative tier: Medial Meniscus (0.030), Lateral OA (0.025), PF OA (0.020), Lateral Meniscus (0.015), Synovitis (0.015)
        # Speed-Demon V10 Specialist Weights: blending our 6-Model Cleanlab-Pruned Multi-Resolution MoE
        # (3x N4 ConvNeXt-Small + 3x C384 CoAtNet, 0.8676 Silver OOF / 0.8568 Gold AUC)
        # directly with Unified Dual-Raptor (0.65 * Raptor-384 + 0.35 * Raptor-224):
        OPTIMIZED_TARGET_WEIGHTS = {
            "Medial OA": 0.0650,
            "Baker's": 0.0650,
            "Fracture": 0.0550,
            "Contusion": 0.0500,
            "ACL": 0.0500,
            "MCL": 0.0500,
            "Effusion": 0.0500,
            "Medial Meniscus": 0.0350,
            "Lateral OA": 0.0300,
            "PF OA": 0.0250,
            "Lateral Meniscus": 0.0150,
            "Synovitis": 0.0150,
        }

        # Verified 0.950 V1 weights between Stage 1 (_sota_0949.csv) and Stage 2 (_own.csv)
        VERIFIED_V1_OWN_WEIGHTS = {
            "Baker's": 0.35, "Contusion": 0.30, "Medial OA": 0.25, "ACL": 0.25,
            "Medial Meniscus": 0.20, "MCL": 0.20, "Lateral Meniscus": 0.08,
            "Fracture": 0.08, "Lateral OA": 0.08, "PF OA": 0.08,
            "Effusion": 0.04, "Synovitis": 0.03,
        }

        blended = pd.DataFrame(index=their_sub.index)
        if os.path.exists('/kaggle/working/_sota_0949.csv') and os.path.exists('/kaggle/working/_own.csv'):
            print("  --> Performing unquantized 3-way continuous rank fusion (Stage 1 SOTA + Stage 2 MIL + Stage 4 Creaky MoE)")
            sota_df = pd.read_csv('/kaggle/working/_sota_0949.csv', dtype={'StudyInstanceUID': str})
            own_df = pd.read_csv('/kaggle/working/_own.csv', dtype={'StudyInstanceUID': str})
            sota_df = sota_df.set_index('StudyInstanceUID').reindex(their_sub['StudyInstanceUID']).reset_index()
            own_df = own_df.set_index('StudyInstanceUID').reindex(their_sub['StudyInstanceUID']).reset_index()
            rank_sota = sota_df[TARGET_COLS].rank(method="average", pct=True)
            rank_own = own_df[TARGET_COLS].rank(method="average", pct=True)
            for col in TARGET_COLS:
                our_w = float(OPTIMIZED_TARGET_WEIGHTS.get(col, 0.03))
                v1_own_w = float(VERIFIED_V1_OWN_WEIGHTS.get(col, 0.15)) * (1.0 - our_w)
                sota_w = 1.0 - v1_own_w - our_w
                blended[col] = sota_w * rank_sota[col] + v1_own_w * rank_own[col] + our_w * our_ranks[col]
                print(f"Target '{col:16s}': {sota_w:.4f} SOTA_384 + {v1_own_w:.4f} MIL_2.5D + {our_w:.4f} Creaky_N4_C384")
            blended[TARGET_COLS] = blended[TARGET_COLS].rank(method="average", pct=True)
        else:
            their_ranks = their_sub[TARGET_COLS].rank(method="average", pct=True)
            for col in TARGET_COLS:
                our_w = float(OPTIMIZED_TARGET_WEIGHTS.get(col, 0.03))
                their_w = 1.0 - our_w
                blended[col] = their_w * their_ranks[col] + our_w * our_ranks[col]
                print(f"Target '{col:16s}': weight {our_w:.4f} Creaky_N4_C384 / {their_w:.4f} foundation")
            blended[TARGET_COLS] = blended[TARGET_COLS].rank(method="average", pct=True)

        their_sub[TARGET_COLS] = blended[TARGET_COLS]
        their_sub.to_csv('submission.csv', index=False)
        print('Blend complete! Wrote final submission.csv')
    except Exception as e:
        print('Blend failed! Outputting original submission.csv just to be safe.')
        print(e)
except Exception as _e:
    print('Our Creaky branch failed; restoring untouched public submission:', _e)
    _sh.copy('public_submission_backup.csv', 'submission.csv')

